In [1]:
# ============================================================
# HYBRID INDOBERT + HANDCRAFTED FEATURE FUSION
# Low-Resource Indonesian Hate Speech Detection
# ============================================================

import os
import re
import math
import random
import warnings
import numpy as np
import pandas as pd

from collections import Counter
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import SequenceClassifierOutput

warnings.filterwarnings("ignore")

# ============================================================
# 1. GLOBALS
# ============================================================

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"
OUTPUT_DIR = "./paper_outputs_hybrid"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")
PATH_ABUSIVE = os.path.join(DATA_DIR, "abusive.csv")

print("Using device:", DEVICE)

# ============================================================
# 2. SEED
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

# ============================================================
# 3. FILE LOADING
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception as e:
            last_error = e
    raise ValueError(f"Could not read file {path}. Last error: {last_error}")

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "not_hs", "0", "false", "no"}:
            return 0

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)

    return np.nan

def print_dataset_summary(df, name="dataset"):
    print("\n" + "=" * 70)
    print(f"DATASET: {name}")
    print("=" * 70)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    if "label" in df.columns:
        print("Label distribution:", Counter(df["label"].tolist()))

def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "IDHSD_713"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df

def load_572_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["comment_text", "tweet", "Tweet", "text", "Text"]
    possible_label_cols = ["class", "label", "Label", "HS"]

    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    label_col = next((c for c in possible_label_cols if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(f"Could not detect text/label columns in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "HS_572"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df

def load_re_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["Tweet", "tweet", "text", "Text"]
    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    if text_col is None:
        raise ValueError(f"No text column found in {path}")

    if "HS" in df.columns:
        label_col = "HS"
    elif "label" in df.columns:
        label_col = "label"
    else:
        raise ValueError(f"No HS/label column found in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "RE_13169"

    keep_cols = ["text", "label", "source"]
    if "Abusive" in df.columns:
        keep_cols.append("Abusive")

    df = df[keep_cols].dropna(subset=["text", "label"]).reset_index(drop=True)
    return df

def merge_datasets(datasets):
    merged = pd.concat(datasets, axis=0, ignore_index=True)
    merged["text"] = merged["text"].astype(str).str.strip()
    merged = merged[merged["text"].str.len() > 0].copy()
    merged = merged.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return merged

# ============================================================
# 4. TEXT NORMALIZATION
# ============================================================

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "wkwwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa",
    "xixi": "tertawa",
}

def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    if df.shape[1] < 2:
        raise ValueError("new_kamusalay.csv must have at least 2 columns.")
    col1, col2 = df.columns[:2]
    slang_dict = {}
    for _, row in df.iterrows():
        src = str(row[col1]).strip().lower()
        tgt = str(row[col2]).strip().lower()
        if src and src != "nan":
            slang_dict[src] = tgt
    return slang_dict

def load_abusive_lexicon(path):
    df = safe_read_csv(path)
    first_col = df.columns[0]
    return set(str(x).strip().lower() for x in df[first_col].dropna().tolist())

def reduce_repeated_chars(text):
    return REPEAT_CHAR_PATTERN.sub(r"\1\1", text)

def split_hashtag(match):
    return f" {match.group(1)} "

def normalize_special_tokens(text):
    tokens = text.split()
    out = []
    for tok in tokens:
        out.append(LAUGHTER_VARIANTS.get(tok, tok))
    return " ".join(out)

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\\n", " ")
    text = text.replace("\\/", "/")
    text = reduce_repeated_chars(text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(split_hashtag, text)
    text = text.lower()
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

def slang_normalize(text, slang_dict):
    tokens = text.split()
    return " ".join([slang_dict.get(tok, tok) for tok in tokens])

def preprocess_text(text, slang_dict=None):
    text = basic_clean_text(text)
    text = normalize_special_tokens(text)
    if slang_dict is not None:
        text = slang_normalize(text, slang_dict)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

# ============================================================
# 5. HANDCRAFTED FEATURES
# ============================================================

def uppercase_ratio(raw_text):
    raw_text = str(raw_text)
    if len(raw_text) == 0:
        return 0.0
    n_upper = sum(1 for c in raw_text if c.isupper())
    n_alpha = sum(1 for c in raw_text if c.isalpha())
    if n_alpha == 0:
        return 0.0
    return n_upper / n_alpha

def repeated_char_count(raw_text):
    raw_text = str(raw_text)
    return len(REPEAT_CHAR_PATTERN.findall(raw_text.lower()))

def laughter_count(clean_text):
    tokens = clean_text.split()
    return sum(1 for tok in tokens if tok == "tertawa")

def build_handcrafted_features(df, abusive_lexicon):
    feats = pd.DataFrame()
    feats["abusive_count"] = df["clean_text"].apply(
        lambda txt: sum(1 for tok in txt.split() if tok in abusive_lexicon)
    )
    feats["word_count"] = df["clean_text"].apply(lambda x: len(str(x).split()))
    feats["char_count"] = df["clean_text"].apply(lambda x: len(str(x)))
    feats["exclamation_count"] = df["text"].apply(lambda x: str(x).count("!"))
    feats["question_count"] = df["text"].apply(lambda x: str(x).count("?"))
    feats["uppercase_ratio"] = df["text"].apply(uppercase_ratio)
    feats["repeated_char_count"] = df["text"].apply(repeated_char_count)
    feats["laughter_count"] = df["clean_text"].apply(laughter_count)
    return feats.astype(float)

# ============================================================
# 6. PREPARE DATA
# ============================================================

def prepare_full_dataset():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)
    abusive_words = load_abusive_lexicon(PATH_ABUSIVE)

    ds_idhsd = load_idhsd_dataset(PATH_IDHSD)
    ds_572 = load_572_dataset(PATH_572)
    ds_re = load_re_dataset(PATH_RE)

    print_dataset_summary(ds_idhsd, "IDHSD")
    print_dataset_summary(ds_572, "572 dataset")
    print_dataset_summary(ds_re, "re_dataset")

    data = merge_datasets([ds_idhsd, ds_572, ds_re])
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].copy()

    print_dataset_summary(
        data[["clean_text", "label"]].rename(columns={"clean_text": "text"}),
        "Merged final dataset"
    )
    return data, abusive_words

def make_train_val_test_split(data, test_size=0.2, val_size=0.1):
    train_df, test_df = train_test_split(
        data,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=data["label"]
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def make_low_label_split(train_df, labeled_fraction=0.05):
    labeled_df, unlabeled_df = train_test_split(
        train_df,
        train_size=labeled_fraction,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )
    return labeled_df.reset_index(drop=True), unlabeled_df.reset_index(drop=True)

# ============================================================
# 7. DATASET WITH TOKENIZED TEXT + NUMERIC FEATURES
# ============================================================

class HybridDataset(Dataset):
    def __init__(self, texts, labels, numeric_features, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=False,
            max_length=max_length
        )
        self.labels = list(labels)
        self.numeric_features = np.asarray(numeric_features, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["numeric_features"] = torch.tensor(self.numeric_features[idx], dtype=torch.float32)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class HybridCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        numeric_features = torch.stack([f["numeric_features"] for f in features])
        labels = torch.stack([f["labels"] for f in features])

        text_features = []
        for f in features:
            d = {k: v for k, v in f.items() if k not in ["numeric_features", "labels"]}
            text_features.append(d)

        batch = self.tokenizer.pad(
            text_features,
            padding=True,
            return_tensors="pt"
        )
        batch["numeric_features"] = numeric_features
        batch["labels"] = labels
        return batch

def build_weighted_sampler(labels):
    class_counts = Counter(labels)
    weights = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = [weights[int(lbl)] for lbl in labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

def get_class_weights(labels):
    counts = Counter(labels)
    total = sum(counts.values())
    n_classes = len(counts)
    weights = {}
    for cls, cnt in counts.items():
        weights[int(cls)] = total / (n_classes * cnt)
    ordered = [weights.get(i, 1.0) for i in sorted(weights.keys())]
    return torch.tensor(ordered, dtype=torch.float)

# ============================================================
# 8. HYBRID MODEL
# ============================================================

class HybridIndoBERTClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_numeric_features,
        num_labels=2,
        numeric_hidden_dim=32,
        fusion_hidden_dim=256,
        dropout=0.2
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.numeric_net = nn.Sequential(
            nn.Linear(num_numeric_features, numeric_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + numeric_hidden_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, num_labels)
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        numeric_features=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None
        )

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0]

        numeric_repr = self.numeric_net(numeric_features)
        fused = torch.cat([pooled, numeric_repr], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None
        )

# ============================================================
# 9. CUSTOM TRAINER
# ============================================================

class WeightedHybridTrainer(Trainer):
    def __init__(self, class_weights=None, train_sampler=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.train_sampler_override = train_sampler

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            class_weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self.train_sampler_override if self.train_sampler_override is not None else None,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True
        )

def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else np.nan,
    }

# ============================================================
# 10. EVALUATION
# ============================================================

def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="model"):
    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_prob is not None:
        try:
            result["roc_auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            result["roc_auc"] = np.nan
    else:
        result["roc_auc"] = np.nan
    return result

def tune_threshold_from_probs(y_true, y_prob):
    best_thr = 0.5
    best_score = -1.0
    for thr in np.arange(0.20, 0.81, 0.02):
        pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, pred, average="macro")
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr, best_score

# ============================================================
# 11. FEATURE SCALING
# ============================================================

def build_scaled_features(train_df, val_df, test_df, unlabeled_df, abusive_words):
    train_feats = build_handcrafted_features(train_df, abusive_words)
    val_feats = build_handcrafted_features(val_df, abusive_words)
    test_feats = build_handcrafted_features(test_df, abusive_words)
    unlabeled_feats = build_handcrafted_features(unlabeled_df, abusive_words)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_feats)
    val_scaled = scaler.transform(val_feats)
    test_scaled = scaler.transform(test_feats)
    unlabeled_scaled = scaler.transform(unlabeled_feats)

    return train_scaled, val_scaled, test_scaled, unlabeled_scaled, scaler

# ============================================================
# 12. TRAIN HYBRID MODEL
# ============================================================

def run_hybrid_indobert_model(
    model_checkpoint,
    train_df,
    val_df,
    test_df,
    train_numeric,
    val_numeric,
    test_numeric,
    run_name,
    max_length=128,
    epochs=50,
    batch_size=8,
    learning_rate=2e-5,
    seed=42
):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    train_dataset = HybridDataset(
        train_df["clean_text"].tolist(),
        train_df["label"].tolist(),
        train_numeric,
        tokenizer,
        max_length=max_length
    )
    val_dataset = HybridDataset(
        val_df["clean_text"].tolist(),
        val_df["label"].tolist(),
        val_numeric,
        tokenizer,
        max_length=max_length
    )
    test_dataset = HybridDataset(
        test_df["clean_text"].tolist(),
        test_df["label"].tolist(),
        test_numeric,
        tokenizer,
        max_length=max_length
    )

    num_numeric_features = train_numeric.shape[1]
    model = HybridIndoBERTClassifier(
        model_name=model_checkpoint,
        num_numeric_features=num_numeric_features,
        num_labels=2,
        numeric_hidden_dim=32,
        fusion_hidden_dim=256,
        dropout=0.2
    )

    class_weights = get_class_weights(train_df["label"].tolist())
    sampler = build_weighted_sampler(train_df["label"].tolist())
    collator = HybridCollator(tokenizer)

    run_output_dir = os.path.join(OUTPUT_DIR, run_name)
    os.makedirs(run_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=run_output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=seed,
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )

    trainer = WeightedHybridTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        compute_metrics=compute_metrics_for_trainer,
        class_weights=class_weights,
        train_sampler=sampler,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=50)],
    )

    trainer.train()

    val_output = trainer.predict(val_dataset)
    val_logits = val_output.predictions
    val_prob = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()
    best_thr, _ = tune_threshold_from_probs(val_df["label"], val_prob)
    val_pred = (val_prob >= best_thr).astype(int)

    test_output = trainer.predict(test_dataset)
    test_logits = test_output.predictions
    test_prob = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()
    test_pred = (test_prob >= best_thr).astype(int)

    print("\n" + "=" * 70)
    print(f"RESULTS: {run_name}")
    print("=" * 70)
    print(classification_report(test_df["label"], test_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(test_df["label"], test_pred))

    val_result = evaluate_predictions(val_df["label"], val_pred, val_prob, model_name=run_name)
    test_result = evaluate_predictions(test_df["label"], test_pred, test_prob, model_name=run_name)

    val_result.update({"split": "val", "best_threshold": best_thr, "seed": seed})
    test_result.update({"split": "test", "best_threshold": best_thr, "seed": seed})

    return pd.DataFrame([val_result, test_result]), trainer, tokenizer, best_thr

# ============================================================
# 13. OPTIONAL: CONTROLLED SSL
# ============================================================

def predict_hybrid_probs(trainer, dataset):
    output = trainer.predict(dataset)
    logits = output.predictions
    return torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

def select_balanced_pseudo_labels(
    unlabeled_df,
    probs,
    round_idx=1,
    base_threshold=0.92,
    threshold_decay=0.05,
    max_add_per_round=1500
):
    current_thr = max(0.70, base_threshold - (round_idx - 1) * threshold_decay)

    preds = (probs >= 0.5).astype(int)
    confident = (probs >= current_thr) | (probs <= 1 - current_thr)

    cand = unlabeled_df.loc[confident].copy()
    if cand.empty:
        return pd.DataFrame(), unlabeled_df.copy(), current_thr

    cand["pseudo_label"] = preds[confident]
    cand["confidence"] = probs[confident]
    cand["confidence_margin"] = np.abs(cand["confidence"] - 0.5)

    per_class_cap = max_add_per_round // 2

    hs_cand = cand[cand["pseudo_label"] == 1].sort_values("confidence_margin", ascending=False).head(per_class_cap)
    non_cand = cand[cand["pseudo_label"] == 0].sort_values("confidence_margin", ascending=False).head(per_class_cap)

    selected = pd.concat([hs_cand, non_cand], ignore_index=True)

    if len(selected) < max_add_per_round:
        remaining = cand.drop(index=selected.index, errors="ignore").sort_values("confidence_margin", ascending=False)
        extra_needed = max_add_per_round - len(selected)
        selected = pd.concat([selected, remaining.head(extra_needed)], ignore_index=True)

    selected_texts = set(selected["clean_text"].tolist())
    remaining_df = unlabeled_df[~unlabeled_df["clean_text"].isin(selected_texts)].reset_index(drop=True)

    return selected.reset_index(drop=True), remaining_df, current_thr

def run_hybrid_ssl(
    labeled_df,
    unlabeled_df,
    val_df,
    test_df,
    abusive_words,
    scenario_name,
    max_rounds=2,
    seed=42
):
    current_labeled = labeled_df.copy().reset_index(drop=True)
    current_unlabeled = unlabeled_df.copy().reset_index(drop=True)
    frames = []

    for round_idx in range(1, max_rounds + 1):
        round_name = f"{scenario_name}_hybrid_ssl_round{round_idx}"

        train_num, val_num, test_num, unlabeled_num, scaler = build_scaled_features(
            current_labeled, val_df, test_df, current_unlabeled, abusive_words
        )

        hybrid_results, hybrid_trainer, hybrid_tokenizer, hybrid_thr = run_hybrid_indobert_model(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=current_labeled,
            val_df=val_df,
            test_df=test_df,
            train_numeric=train_num,
            val_numeric=val_num,
            test_numeric=test_num,
            run_name=f"hybrid_indobert_{round_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=seed
        )
        frames.append(hybrid_results)

        pred_dataset = HybridDataset(
            current_unlabeled["clean_text"].tolist(),
            [0] * len(current_unlabeled),
            unlabeled_num,
            hybrid_tokenizer,
            max_length=128
        )

        probs = predict_hybrid_probs(hybrid_trainer, pred_dataset)

        pseudo_df, current_unlabeled, used_thr = select_balanced_pseudo_labels(
            current_unlabeled,
            probs,
            round_idx=round_idx,
            base_threshold=0.92,
            threshold_decay=0.05,
            max_add_per_round=max(1000, int(len(current_labeled) * 2))
        )

        print("\n" + "#" * 70)
        print(f"HYBRID SSL ROUND: {round_name}")
        print("#" * 70)
        print("Used pseudo-label threshold:", used_thr)
        print("Pseudo-labeled added:", len(pseudo_df))
        print("Remaining unlabeled:", len(current_unlabeled))

        if pseudo_df.empty:
            break

        pseudo_add = pseudo_df.copy()
        if "label" in pseudo_add.columns:
            pseudo_add = pseudo_add.drop(columns=["label"])
        pseudo_add["label"] = pseudo_df["pseudo_label"].astype(int)
        pseudo_add = pseudo_add.reindex(columns=current_labeled.columns)

        current_labeled = pd.concat([current_labeled, pseudo_add], ignore_index=True)
        current_labeled = current_labeled.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ============================================================
# 14. MAIN PIPELINE
# ============================================================

def run_hybrid_pipeline():
    data, abusive_words = prepare_full_dataset()
    train_df, val_df, test_df = make_train_val_test_split(data, test_size=0.2, val_size=0.1)

    scenarios = [0.05, 0.10, 0.20]
    all_results = []

    for frac in scenarios:
        scenario_name = f"{int(frac * 100)}pct"
        print("\n" + "=" * 80)
        print(f"LOW-LABEL SCENARIO: {scenario_name}")
        print("=" * 80)

        labeled_df, unlabeled_df = make_low_label_split(train_df, labeled_fraction=frac)

        train_num, val_num, test_num, unlabeled_num, scaler = build_scaled_features(
            labeled_df, val_df, test_df, unlabeled_df, abusive_words
        )

        # Hybrid IndoBERT baseline
        hybrid_results, hybrid_trainer, hybrid_tokenizer, hybrid_thr = run_hybrid_indobert_model(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=labeled_df,
            val_df=val_df,
            test_df=test_df,
            train_numeric=train_num,
            val_numeric=val_num,
            test_numeric=test_num,
            run_name=f"hybrid_indobert_{scenario_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=42
        )
        hybrid_results["label_fraction"] = frac
        all_results.append(hybrid_results)

        # Optional SSL
        hybrid_ssl_results = run_hybrid_ssl(
            labeled_df=labeled_df,
            unlabeled_df=unlabeled_df,
            val_df=val_df,
            test_df=test_df,
            abusive_words=abusive_words,
            scenario_name=scenario_name,
            max_rounds=2,
            seed=42
        )
        if not hybrid_ssl_results.empty:
            hybrid_ssl_results["label_fraction"] = frac
            all_results.append(hybrid_ssl_results)

    final_results = pd.concat(all_results, axis=0, ignore_index=True)
    final_results.to_csv(os.path.join(OUTPUT_DIR, "hybrid_results.csv"), index=False)

    print("\nTop TEST results by macro-F1:")
    print(
        final_results[final_results["split"] == "test"]
        .sort_values("macro_f1", ascending=False)
        [["label_fraction", "model", "accuracy", "macro_f1", "macro_precision", "macro_recall", "roc_auc", "best_threshold", "seed"]]
        .head(30)
        .to_string(index=False)
    )

    return final_results

# ============================================================
# 15. ENTRY
# ============================================================

if __name__ == "__main__":
    results = run_hybrid_pipeline()

2026-04-19 05:00:07.885303: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda

DATASET: IDHSD
Shape: (713, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({0: 453, 1: 260})

DATASET: 572 dataset
Shape: (572, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({1: 286, 0: 286})

DATASET: re_dataset
Shape: (13169, 4)
Columns: ['text', 'label', 'source', 'Abusive']
Label distribution: Counter({0: 7608, 1: 5561})

DATASET: Merged final dataset
Shape: (14184, 2)
Columns: ['text', 'label']
Label distribution: Counter({0: 8251, 1: 5933})

LOW-LABEL SCENARIO: 5pct


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.695500,0.688365,0.490749,0.478348,0.537495,0.529960,0.574470
2,0.674200,0.673573,0.471366,0.396824,0.667859,0.543094,0.736016
3,0.625500,0.652042,0.488987,0.422544,0.697103,0.559131,0.811663
4,0.575400,0.575028,0.718062,0.717245,0.759059,0.746659,0.857531
5,0.446800,0.487753,0.776211,0.775282,0.779342,0.786627,0.871493
6,0.304300,0.460649,0.792070,0.789434,0.788181,0.794952,0.871400
7,0.167900,0.487858,0.795595,0.790500,0.789887,0.791196,0.869088
8,0.114600,0.628826,0.782379,0.763072,0.798867,0.755933,0.849276
9,0.094300,0.680650,0.744493,0.744460,0.768939,0.766730,0.864901
10,0.076400,0.716798,0.783260,0.764670,0.798241,0.757576,0.845225



RESULTS: hybrid_indobert_5pct
              precision    recall  f1-score   support

           0     0.8569    0.7873    0.8206      1650
           1     0.7343    0.8172    0.7735      1187

    accuracy                         0.7998      2837
   macro avg     0.7956    0.8022    0.7971      2837
weighted avg     0.8056    0.7998    0.8009      2837

Confusion Matrix:
[[1299  351]
 [ 217  970]]


...


######################################################################
HYBRID SSL ROUND: 20pct_hybrid_ssl_round1
######################################################################
Used pseudo-label threshold: 0.92
Pseudo-labeled added: 4084
Remaining unlabeled: 4086


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.623700,0.550431,0.742731,0.742395,0.752204,0.756954,0.843684
2,0.251600,0.401881,0.841410,0.838235,0.836480,0.840917,0.909356
3,0.125300,0.487042,0.833480,0.832606,0.835103,0.844131,0.918029
4,0.087600,0.476019,0.857269,0.853109,0.853781,0.852488,0.920077
5,0.074300,0.606324,0.841410,0.840144,0.840265,0.849474,0.919100
6,0.070300,0.533465,0.854626,0.851289,0.850024,0.852871,0.919987
7,0.039200,0.591624,0.857269,0.853711,0.852959,0.854553,0.922105
8,0.027900,0.695755,0.842291,0.839711,0.837676,0.844035,0.916944
9,0.020300,0.749510,0.851982,0.850655,0.850129,0.859450,0.925703
10,0.018900,0.829234,0.846696,0.840655,0.845981,0.837201,0.917608



RESULTS: hybrid_indobert_20pct_hybrid_ssl_round2
              precision    recall  f1-score   support

           0     0.8766    0.8479    0.8620      1650
           1     0.7977    0.8340    0.8155      1187

    accuracy                         0.8421      2837
   macro avg     0.8372    0.8410    0.8387      2837
weighted avg     0.8436    0.8421    0.8425      2837

Confusion Matrix:
[[1399  251]
 [ 197  990]]



######################################################################
HYBRID SSL ROUND: 20pct_hybrid_ssl_round2
######################################################################
Used pseudo-label threshold: 0.87
Pseudo-labeled added: 4082
Remaining unlabeled: 134

Top TEST results by macro-F1:
 label_fraction                                   model  accuracy  macro_f1  macro_precision  macro_recall  roc_auc  best_threshold  seed
           0.20 hybrid_indobert_20pct_hybrid_ssl_round2  0.842087  0.838735         0.837155      0.840957 0.914703            0.80    42
           0.20 hybrid_indobert_20pct_hybrid_ssl_round1  0.835037  0.831213         0.830060      0.832651 0.911343            0.76    42
           0.20                   hybrid_indobert_20pct  0.835037  0.831213         0.830060      0.832651 0.911343            0.76    42
           0.10                   hybrid_indobert_10pct  0.828692  0.824529         0.823664      0.825541 0.897288            0.74    42
        